# YOLO Pose Estimation Example

## YOLO Pose Estimation

This notebook is an example of human pose estimation with **Ultralytics YOLO**.

```
Image (URL or file) → YOLO pose estimator → Annotated image + keypoints + confidence scores
```

Unlike detection (bounding box only), pose estimation **localizes body keypoints** (e.g. eyes, shoulders, elbows, wrists, hips, knees, ankles) for each detected person.

### Available models

| Model | Params | Speed |
|-------|--------|-------|
| `yolo11x-pose.pt` | 58.8 M | slowest / most accurate |
| `yolo11l-pose.pt` | 26.2 M | — |
| `yolo11m-pose.pt` | 20.9 M | — |
| `yolo11s-pose.pt` |  9.9 M | — |
| `yolo11n-pose.pt` |  2.9 M | fastest / lightest |

Change `MODEL_NAME` in the writefile cell to switch between them.

📄 [Ultralytics YOLO docs](https://docs.ultralytics.com/)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

PROJECT_PATH = "/content/drive/MyDrive/CV-Samples/yolo"
!mkdir -p "{PROJECT_PATH}"
%cd "{PROJECT_PATH}"
!ls

In [ ]:
# Install Ultralytics
!pip install ultralytics -q

import ultralytics
ultralytics.checks()  # check environment

In [ ]:
# Download sample images from GitHub
import os

# Image files to download
IMAGE_FILES = [
    'lograstudio-yoga-3053487_640.jpg',
]

BASE_URL = 'https://raw.githubusercontent.com/mastnk/cv-samples/main/yolo'
for fname in IMAGE_FILES:
    url = f'{BASE_URL}/{fname}'
    dest = f'{PROJECT_PATH}/{fname}'
    if not os.path.exists(dest):
        os.system(f'wget -q "{url}" -O "{dest}"')
        print(f'Downloaded: {fname}')
    else:
        print(f'Already exists: {fname}')

%cd "{PROJECT_PATH}"
!ls


## Using Your Own Images

There are two ways to provide images for pose estimation:

**① Specify a URL**  
Pass a direct image URL to the `--url` flag when running the script:
```
%run pose.py --url https://cdn.pixabay.com/photo/2015/08/10/18/01/dancing-882940_640.jpg
```

**② Upload images to the `PROJECT_PATH/` folder**  
Place your image files in `PROJECT_PATH/`, then run the script with `--file` or `--dir`.

The easiest way to upload is through **Google Drive**:  
Open [drive.google.com](https://drive.google.com), navigate to `My Drive / CV-Samples / yolo/`, and drag & drop your files there.  
They will be immediately available in Colab via the mounted drive — no extra upload step needed.

```
%run pose.py --file your_image.jpg   # single file
%run pose.py --dir  .                # all images in folder
```

## Selecting a Model

In the writefile cell below, change `MODEL_NAME` to switch models.  
When multiple `MODEL_NAME = ...` lines are listed, **only the last one takes effect**.

```python
# MODEL_NAME = 'yolo11x-pose.pt'  # 58.8M params — most accurate, slowest
# MODEL_NAME = 'yolo11l-pose.pt'  # 26.2M params
# MODEL_NAME = 'yolo11m-pose.pt'  # 20.9M params
# MODEL_NAME = 'yolo11s-pose.pt'  #  9.9M params
MODEL_NAME   = 'yolo11n-pose.pt'  #  2.9M params — fastest, lightest  ← active
```

Larger models are more accurate but take longer to run. Start with `yolo11n-pose.pt` for quick experiments.

In [ ]:
# Save this cell as a Python file (Execute after editing)
%%writefile pose.py
"""YOLO Pose Estimation — command-line interface.

Usage:
  %run pose.py --url  <image_url>  [--disp] [--conf FLOAT]
  %run pose.py --file <image_path> [--disp] [--conf FLOAT]
  %run pose.py --dir  <image_dir>  [--disp] [--conf FLOAT]
"""

import argparse
import glob
import os

from ultralytics import YOLO
from PIL import Image
import requests
from io import BytesIO

# ---- IPython display (works when executed via %run in Colab) -----
try:
    from IPython.display import display as ipy_display
    _has_ipy = True
except ImportError:
    _has_ipy = False

# ---- Configuration -----------------------------------------------
MODEL_NAME   = 'yolo11n-pose.pt'  #  2.9M params — fastest / lightest
# MODEL_NAME = 'yolo11s-pose.pt'  #  9.9M params
# MODEL_NAME = 'yolo11m-pose.pt'  # 20.9M params
# MODEL_NAME = 'yolo11l-pose.pt'  # 26.2M params
# MODEL_NAME = 'yolo11x-pose.pt'  # 58.8M params — most accurate

PROJECT_PATH = '/content/drive/MyDrive/CV-Samples/yolo'

# ---- Model loading -----------------------------------------------
model = YOLO(MODEL_NAME)

# ---- Display helper ----------------------------------------------
def show(annotated, label: str, disp: bool) -> None:
    """When --disp is active, display the annotated image then print the filename/URL."""
    if disp:
        if _has_ipy:
            ipy_display(annotated)
        print(label)

# ---- Functions ---------------------------------------------------
def pose_url(url: str, conf: float = 0.25, disp: bool = False):
    """Download an image from a URL and estimate pose."""
    image     = Image.open(BytesIO(requests.get(url).content)).convert('RGB')
    results   = model.predict(image, conf=conf, verbose=False)
    annotated = Image.fromarray(results[0].plot()[:, :, ::-1])  # BGR -> RGB
    show(annotated, url, disp)
    boxes = results[0].boxes
    names = results[0].names
    for box in boxes:
        cls_id   = int(box.cls[0])
        conf_val = float(box.conf[0])
        print(f'  {names[cls_id]:<20} {conf_val*100:.1f}%')
    return results


def pose_file(path: str, conf: float = 0.25, disp: bool = False):
    """Estimate pose in a single local image file."""
    image     = Image.open(path).convert('RGB')
    results   = model.predict(image, conf=conf, verbose=False)
    annotated = Image.fromarray(results[0].plot()[:, :, ::-1])  # BGR -> RGB
    show(annotated, path, disp)
    boxes = results[0].boxes
    names = results[0].names
    for box in boxes:
        cls_id   = int(box.cls[0])
        conf_val = float(box.conf[0])
        print(f'  {names[cls_id]:<20} {conf_val*100:.1f}%')
    return results


def pose_dir(directory: str, conf: float = 0.25, disp: bool = False):
    """Estimate pose in all images in a directory."""
    exts = ['jpg', 'jpeg', 'png', 'bmp', 'JPG', 'JPEG', 'PNG', 'BMP']
    filepaths = []
    for ext in exts:
        filepaths += sorted(glob.glob(os.path.join(directory, f'*.{ext}')))

    if not filepaths:
        print(f'No images found in: {directory}')
        return

    for path in filepaths:
        pose_file(path, conf, disp)
        print()


# ---- Argument parsing --------------------------------------------
parser = argparse.ArgumentParser(description='YOLO Pose Estimation')
group  = parser.add_mutually_exclusive_group(required=True)
group.add_argument('--url',  type=str,              help='Image URL for pose estimation')
group.add_argument('--file', type=str,              help='Path to a single image file')
group.add_argument('--dir',  type=str,              help='Directory of images')
parser.add_argument('--conf', type=float, default=0.25, help='Confidence threshold (default: 0.25)')
parser.add_argument('--disp', action='store_true',      help='Display annotated image and filename/URL before results')
args = parser.parse_args()

# ---- Run ---------------------------------------------------------
if args.url:
    pose_url(args.url,   conf=args.conf, disp=args.disp)
elif args.file:
    pose_file(args.file, conf=args.conf, disp=args.disp)
elif args.dir:
    pose_dir(args.dir,   conf=args.conf, disp=args.disp)


## `pose.py` Usage

After running the `%%writefile` cell above, `pose.py` is saved in `PROJECT_PATH`.
Run it with **`%run`** (not `!python`) so that inline image display works in Colab.

```
%run pose.py --url  <image_url>        # estimate pose in a remote image
%run pose.py --file <image_path>       # estimate pose in a single local file
%run pose.py --dir  <image_directory>  # estimate pose in all images in a folder
```

**Optional arguments**

| Flag | Default | Description |
|------|---------|-------------|
| `--disp` | off | Display annotated image and filename / URL before results |
| `--conf <f>` | `0.25` | Confidence threshold for detections (0.0 – 1.0) |

**Examples**

```bash
# Estimate pose in a remote image and display it
%run pose.py --url https://cdn.pixabay.com/photo/2015/08/10/18/01/dancing-882940_640.jpg --disp

# Estimate pose in one file, lower confidence threshold
%run pose.py --file lograstudio-yoga-3053487_640.jpg --conf 0.1 --disp

# Estimate pose in every image in the folder, display each one
%run pose.py --dir . --disp
```

**Output format (with `--disp`)**

```
<annotated image with keypoints displayed inline>
lograstudio-yoga-3053487_640.jpg
  person               91.2%
```

## Execution Methods

Use **`%run`** to execute `pose.py` inside the Colab kernel — this enables inline image display with `--disp`.

| Mode | Flag | Description |
|------|------|-------------|
| From URL | `--url <url>` | Fetches and estimates pose (supports Pixabay page URLs) |
| Single file | `--file <path>` | Estimates pose in one local image |
| Directory | `--dir <path>` | Estimates pose in all images in a folder |

Add `--disp` to display each annotated image (with keypoints) and its filename / URL before the results.  
Add `--conf <f>` to change the confidence threshold (default: 0.25).

In [ ]:
# Execute: estimate pose in an image from URL (with display)
%cd "{PROJECT_PATH}"
%run pose.py --disp --url https://cdn.pixabay.com/photo/2015/08/10/18/01/dancing-882940_640.jpg


In [ ]:
# Execute: estimate pose in a single local image file (with display)
%cd "{PROJECT_PATH}"
%run pose.py --disp --file lograstudio-yoga-3053487_640.jpg


In [ ]:
# Execute: estimate pose in all images in a directory (with display)
%cd "{PROJECT_PATH}"
%run pose.py --disp --dir .


💾 **Don't forget to save this notebook!**

To keep your work, go to **File → Save a copy in Drive** before closing.

